In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

import os
import imageio.v2 as imageio
from IPython.display import display, Video

import sys
sys.path.extend(["../../../"])
from utils import computation_graph_linear, grad_squared_loss_wrt_linear_model, squared_loss_function

# Stochastic Gradient Descent

In [ ]:
try:
    plt.close("all")
except:
    pass
    
# 1. Dataset
np.random.seed(42)
N = 10000
X = 2 * np.random.rand(N)
true_w, true_b = np.array([[3.0]]), np.array([[1.0]])
y = (computation_graph_linear(X.reshape(-1, 1), true_w, true_b) + np.random.randn(N, 1) * 0.5).ravel()

# 2. Cost grid
def compute_cost_grid(W, B, X, y):
    X_expanded = X[np.newaxis, :, np.newaxis]
    y_expanded = y[np.newaxis, :, np.newaxis]
    W_expanded = W.reshape(-1, 1, 1)
    B_expanded = B.reshape(-1, 1, 1)
    y_pred = computation_graph_linear(X_expanded, W_expanded, B_expanded)
    cost = np.mean(squared_loss_function(y_expanded, y_pred), axis=(1, 2))
    return cost.reshape(W.shape)

w_vals = np.linspace(-1, 5, 100)
b_vals = np.linspace(-1, 5, 100)
W, B = np.meshgrid(w_vals, b_vals)
Z = compute_cost_grid(W, B, X, y)

# 3. Init
lr = 0.25
steps = 20
batch_size = 100

# --- GD ---
w_gd, b_gd = np.array([[0.2]]), np.array([[4.0]])
path_gd = [(w_gd[0, 0], b_gd[0, 0])]

start = time.time()
for i in range(steps):
    grad_w, grad_b = grad_squared_loss_wrt_linear_model(X.reshape(-1, 1), y.reshape(-1, 1), w_gd, b_gd)
    w_gd = w_gd - (lr / N) * grad_w
    b_gd = b_gd - (lr / N) * grad_b
    path_gd.append((w_gd[0, 0], b_gd[0, 0]))
time_budget = time.time() - start

# --- Mini-batch SGD ---
w_sgd, b_sgd = np.array([[0.2]]), np.array([[4.0]])
path_sgd = [(w_sgd[0, 0], b_sgd[0, 0])]

start = time.time()
break_now = False
while True:
    indices = np.random.permutation(N)
    for i in range(0, N, batch_size):
        batch_idx = indices[i:i+batch_size]
        Xb = X[batch_idx]
        yb = y[batch_idx]

        grad_w, grad_b = grad_squared_loss_wrt_linear_model(Xb.reshape(-1, 1), yb.reshape(-1, 1), w_sgd, b_sgd)
        w_sgd = w_sgd - (lr / batch_size) * grad_w
        b_sgd = b_sgd - (lr / batch_size) * grad_b

        path_sgd.append((w_sgd[0, 0], b_sgd[0, 0]))

        if time.time() - start > time_budget:
            break_now = True
            break
    if break_now:
        break

print(time.time() - start)
print(time_budget)

# Convert
path_gd = np.array(path_gd)
path_sgd = np.array(path_sgd)

# 4. Animación
fig, ax = plt.subplots(figsize=(10, 6))

contours = ax.contour(W, B, Z, levels=30)
ax.clabel(contours, inline=True, fontsize=8)

line_gd, = ax.plot([], [], 'o-', label='GD')
line_sgd, = ax.plot([], [], 'x-', label='Mini-batch SGD')

ax.set_xlabel('w')
ax.set_ylabel('b')
ax.set_title('GD vs Mini-batch SGD')
ax.legend()

max_len = max(len(path_gd), len(path_sgd))

video_filename = "/tmp/aux.mp4"
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

for i in range(1, max_len):
    if i < len(path_gd):
        line_gd.set_data(path_gd[:i, 0], path_gd[:i, 1])
    if i < len(path_sgd):
        line_sgd.set_data(path_sgd[:i, 0], path_sgd[:i, 1])

    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

writer.close()
plt.close(fig)


In [ ]:
display(Video(data=video_filename, embed=True))
os.remove(video_filename)